# Clinical Reasoning | reproducible model comparison

**Objective:** generate a clinical response from each case `Prompt`, using the human `Clinician` answer as the training reference. This is a research competition workflow, not a clinical decision system. No clinical safety or factual correctness claim follows from automated text scores.

**Run order:** install dependencies in your own environment; put the four CSVs in `DATA_DIR`; run setup and data checks; choose a single `MODEL_KEY`; train it; evaluate on the held-out split; optionally create a test prediction CSV. Training and downloads are never triggered by importing the setup cells. Only train and test CSVs are required; raw files are optional inspection material. The original experimentation notes are preserved and clarified below.

**Resources:** large models may need multiple GPUs or substantial memory. Each run uses its own output directory. Gated repositories require accepting their terms. Set `HF_TOKEN` in the environment or a private Colab secret if authentication is necessary; do not paste credentials into this notebook.

## Environment and configuration

Suggested Python 3.10+ environment: `pip install "transformers>=4.51,<5" "peft>=0.13,<1" "datasets>=3,<5" "accelerate>=1,<2" "sentencepiece>=0.2,<1" "sacremoses>=0.1,<1" pandas numpy torch`. Install a compatible PyTorch build for your GPU separately. This notebook uses Transformers 4.x APIs; freeze exact resolved versions for a competition rerun. No package installation or authentication occurs automatically.

In [ ]:
from __future__ import annotations

import gc
import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from peft import LoraConfig, PeftConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
    Trainer, TrainingArguments, set_seed,
)

SEED = 42
set_seed(SEED)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
DATA_DIR = Path(os.environ.get("CLINICAL_DATA_DIR", ".")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("CLINICAL_OUTPUT_DIR", "./outputs")).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VAL_FRACTION = 0.15
MAX_SOURCE_TOKENS = 768
MAX_TARGET_TOKENS = 384
GENERATION_TOKENS = 384
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
EPOCHS = 2
LEARNING_RATE = 2e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_TOKEN = os.environ.get("HF_TOKEN") or None  # Optional; no login or token printing.
print(f"Data: {DATA_DIR}; outputs: {OUTPUT_DIR}; device: {DEVICE}")

## Data contract and leakage checks

Use `train.csv`/`test.csv` as the consistent modeling inputs. `train_raw.csv`/`test_raw.csv` are optional audit files; do not combine raw and processed rows or use the GPT4.0/LLAMA/GEMINI columns as target answers. An ID is used only for joining submissions. Split before tokenization and build any retrieval index only from the training partition. A clinician label is mandatory for supervised training.

In [ ]:
def read_data(data_dir: Path = DATA_DIR):
    paths = {name: data_dir / f"{name}.csv" for name in ("train", "test", "train_raw", "test_raw")}
    missing = [str(paths[k]) for k in ("train", "test") if not paths[k].is_file()]
    if missing:
        raise FileNotFoundError(f"Missing required CSVs: {missing}. Set CLINICAL_DATA_DIR.")
    frames = {k: pd.read_csv(p) if p.is_file() else None for k, p in paths.items()}
    train, test = frames["train"], frames["test"]
    for name, frame, columns in (
        ("train", train, {"Master_Index", "Prompt", "Clinician"}),
        ("test", test, {"Master_Index", "Prompt"}),
    ):
        absent = columns.difference(frame.columns)
        if absent:
            raise ValueError(f"{name}.csv missing columns: {sorted(absent)}")
        if frame["Master_Index"].isna().any() or frame["Master_Index"].duplicated().any():
            raise ValueError(f"{name}.csv has null or duplicate Master_Index values")
        for col in columns - {"Master_Index"}:
            if frame[col].isna().any() or frame[col].astype(str).str.strip().eq("").any():
                raise ValueError(f"{name}.csv has blank {col} values")
    if set(train.Master_Index) & set(test.Master_Index):
        raise ValueError("Train and test IDs overlap")
    return frames


def split_train_validation(frame: pd.DataFrame, fraction: float = VAL_FRACTION):
    if not 0 < fraction < 1 or len(frame) < 2:
        raise ValueError("Need at least two training rows and 0 < validation fraction < 1")
    groups = frame.Prompt.astype(str).str.strip().str.casefold()
    unique_groups = groups.drop_duplicates().sample(frac=1, random_state=SEED)
    if len(unique_groups) < 2:
        raise ValueError("Need at least two distinct prompts to hold out validation")
    n_val = min(len(unique_groups) - 1, max(1, round(fraction * len(unique_groups))))
    in_val = groups.isin(set(unique_groups.iloc[:n_val]))
    training, validation = frame.loc[~in_val].copy(), frame.loc[in_val].copy()
    return training.reset_index(drop=True), validation.reset_index(drop=True)


frames = read_data()
train_df, val_df = split_train_validation(frames["train"])
test_df = frames["test"]
print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})
display(train_df[["Master_Index", "Clinical Panel"]].head() if "Clinical Panel" in train_df else train_df[["Master_Index"]].head())

## Model registry and research notes

| Key | Checkpoint | Role / access |
| --- | --- | --- |
| `med42_v2` | `m42-health/Llama3-Med42-8B` | Instruction model; can optionally produce training-only distillation targets. |
| `biomedlm` | `stanford-crfm/BioMedLM` | Biomedical causal language model. |
| `chatdoctor` | `zl111/ChatDoctor` | Gated conversational model; requires repository access. |
| `biogpt` | `microsoft/biogpt` | Small biomedical causal baseline; uses SentencePiece/Sacremoses. |
| `phi4_medqa` | `Vijayendra/Phi4-MedQA` | Existing PEFT adapter on a Phi-4 base; adapter base is read from its configuration. |
| `meditron` | `epfl-llm/meditron-7b` | Gated base model; needs instruction tuning on the provided answers. |
| `clinical_t5` | `hossboll/clinical-t5` | Clinical summarization model, adapted here to sequence-to-sequence answers. |
| `gemma3_1b` | `google/gemma-3-1b-it` | Recovered from the original experiment; may require repository access. |
| `gatortron` | `UFNLP/gatortron-base` | Encoder: supervised clinical-panel classifier plus nearest-answer retrieval, never free-text generation. |
| `medpalm` | Google Med-PaLM | Research/managed model with no downloadable Med-PaLM weights; no local training or distillation implementation is claimed. |

Earlier BioGPT and MediTron trials have no surviving training records. The old notes also proposed generic data augmentation, multi-head latent attention and feature-based training; these are research ideas, not recovered experiments. This implementation adds optional teacher distillation, LoRA fine tuning and an encoder-feature baseline. It does not invent Med-PaLM weights, unsupported attention modifications or synthetic clinical labels.

In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    model_id: str
    architecture: str  # causal, seq2seq, adapter, or encoder


MODELS = {
    "med42_v2": ModelSpec("m42-health/Llama3-Med42-8B", "causal"),
    "biomedlm": ModelSpec("stanford-crfm/BioMedLM", "causal"),
    "chatdoctor": ModelSpec("zl111/ChatDoctor", "causal"),
    "biogpt": ModelSpec("microsoft/biogpt", "causal"),
    "phi4_medqa": ModelSpec("Vijayendra/Phi4-MedQA", "adapter"),
    "meditron": ModelSpec("epfl-llm/meditron-7b", "causal"),
    "clinical_t5": ModelSpec("hossboll/clinical-t5", "seq2seq"),
    "gemma3_1b": ModelSpec("google/gemma-3-1b-it", "causal"),
    "gatortron": ModelSpec("UFNLP/gatortron-base", "encoder"),
}
MODEL_KEY = "biogpt"  # Change to one key above; run one model at a time.
assert MODEL_KEY in MODELS


def prompt_for(tokenizer, text: str, architecture: str) -> str:
    instruction = (
        "Read the clinical scenario. Give a concise case summary, then answer each "
        "question in the prompt. State uncertainty when the case lacks information. "
        "Do not invent observations.\n\nScenario and questions:\n" + str(text).strip()
    )
    if architecture in ("causal", "adapter") and tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": instruction}],
            tokenize=False, add_generation_prompt=True,
        )
    return f"Clinical question answering:\n{instruction}\n\nClinical response:\n"


def load_model(key: str, *, for_training: bool = False, use_lora: bool = True):
    if key not in MODELS:
        raise KeyError(f"Unknown model: {key}; valid: {list(MODELS)}")
    spec = MODELS[key]
    if spec.architecture == "encoder":
        raise ValueError("Use the GatorTron classifier/retriever workflow below")
    if spec.architecture == "adapter":
        adapter_config = PeftConfig.from_pretrained(spec.model_id, token=HF_TOKEN)
        base_id = adapter_config.base_model_name_or_path
        if not base_id:
            raise ValueError("Phi4-MedQA adapter does not declare a base model")
        model_id = base_id  # This adapter currently refers to an Unsloth 4-bit base.
    else:
        model_id = spec.model_id
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model_class = AutoModelForSeq2SeqLM if spec.architecture == "seq2seq" else AutoModelForCausalLM
    model = model_class.from_pretrained(model_id, token=HF_TOKEN, torch_dtype="auto")
    if spec.architecture == "adapter":
        model = PeftModel.from_pretrained(model, spec.model_id, token=HF_TOKEN, is_trainable=for_training)
        # Existing adapter is trainable as-is; don't stack a second adapter.
    elif for_training and use_lora:
        task = TaskType.SEQ_2_SEQ_LM if spec.architecture == "seq2seq" else TaskType.CAUSAL_LM
        model = get_peft_model(model, LoraConfig(
            task_type=task, r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules="all-linear", bias="none",
        ))
    return tokenizer, model

## Supervised preparation and training

The causal collator masks padding and prompt tokens (`-100`), so loss is computed on the clinician answer only. The sequence-to-sequence collator pads source and target independently. Long prompts and answers are truncated at the configured token limits; inspect coverage for your data. Use a held-out partition for selection. Existing Phi4-MedQA adapter weights are continued, and each other generative checkpoint receives a fresh LoRA adapter by default. Hardware and repository access determine which runs are feasible.

In [ ]:
def prepare_examples(frame: pd.DataFrame, tokenizer, architecture: str):
    examples = []
    for row in frame.itertuples(index=False):
        prompt = prompt_for(tokenizer, row.Prompt, architecture)
        if architecture == "seq2seq":
            source = tokenizer(prompt, truncation=True, max_length=MAX_SOURCE_TOKENS)
            target = tokenizer(text_target=str(row.Clinician), truncation=True, max_length=MAX_TARGET_TOKENS)
            source["labels"] = target["input_ids"]
            examples.append(source)
        else:
            prompt_ids = tokenizer(prompt, add_special_tokens=False, truncation=True,
                                   max_length=MAX_SOURCE_TOKENS)["input_ids"]
            answer_ids = tokenizer(str(row.Clinician), add_special_tokens=False, truncation=True,
                                   max_length=MAX_TARGET_TOKENS - 1)["input_ids"]
            eos = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
            ids = prompt_ids + answer_ids + eos
            examples.append({"input_ids": ids, "attention_mask": [1] * len(ids),
                             "labels": [-100] * len(prompt_ids) + answer_ids + eos})
    return Dataset.from_list(examples)


def causal_collator(tokenizer):
    def collate(features):
        max_length = max(len(f["input_ids"]) for f in features)
        pad = tokenizer.pad_token_id
        return {name: torch.tensor([
            f[name] + [value] * (max_length - len(f[name])) for f in features
        ], dtype=torch.long) for name, value in (
            ("input_ids", pad), ("attention_mask", 0), ("labels", -100)
        )}
    return collate


def training_arguments(run_dir: Path):
    return TrainingArguments(
        output_dir=str(run_dir), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE, evaluation_strategy="epoch", save_strategy="epoch",
        save_total_limit=2, logging_steps=20, report_to="none", seed=SEED,
        fp16=DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported(),
        bf16=DEVICE.type == "cuda" and torch.cuda.is_bf16_supported(),
        remove_unused_columns=False,
    )


def train_generator(key: str, training: pd.DataFrame = train_df,
                    validation: pd.DataFrame = val_df):
    spec = MODELS[key]
    if spec.architecture == "encoder":
        raise ValueError("Run train_gatortron instead")
    tokenizer, model = load_model(key, for_training=True)
    train_set = prepare_examples(training, tokenizer, spec.architecture)
    val_set = prepare_examples(validation, tokenizer, spec.architecture)
    collator = (DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
                if spec.architecture == "seq2seq" else causal_collator(tokenizer))
    run_dir = OUTPUT_DIR / key
    trainer = Trainer(model=model, args=training_arguments(run_dir),
                      train_dataset=train_set, eval_dataset=val_set,
                      data_collator=collator, processing_class=tokenizer)
    trainer.train()
    trainer.save_model(str(run_dir / "final"))
    tokenizer.save_pretrained(run_dir / "final")
    return tokenizer, model, trainer


# Explicit training entry point (uncomment after setting MODEL_KEY and checking resources):
# tokenizer, model, trainer = train_generator(MODEL_KEY)

## Inference, validation and submission

The inference function removes causal prompt tokens by token IDs, not by string length. For T5 it decodes the generated output directly. The evaluation reports exact match and token F1 against clinician text; these are reference similarity measures, not clinical correctness. Review clinical accuracy separately before interpreting results.

In [ ]:
def generate_answers(key: str, tokenizer, model, prompts, batch_size: int = 1):
    spec = MODELS[key]
    if spec.architecture == "encoder":
        raise ValueError("Use predict_gatortron for the encoder")
    model.eval()
    device = next(model.parameters()).device
    responses = []
    for start in range(0, len(prompts), batch_size):
        batch = list(prompts[start:start + batch_size])
        rendered = [prompt_for(tokenizer, text, spec.architecture) for text in batch]
        inputs = tokenizer(rendered, padding=True, truncation=True,
                           max_length=MAX_SOURCE_TOKENS, return_tensors="pt").to(device)
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=GENERATION_TOKENS,
                                       do_sample=False, pad_token_id=tokenizer.pad_token_id)
        if spec.architecture == "seq2seq":
            responses.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        else:
            # Padded prompt width is the first generated-token offset for every row.
            responses.extend(tokenizer.batch_decode(generated[:, inputs["input_ids"].shape[1]:],
                                                    skip_special_tokens=True))
    return [answer.strip() for answer in responses]


def token_f1(prediction: str, reference: str) -> float:
    from collections import Counter
    a, b = Counter(prediction.lower().split()), Counter(reference.lower().split())
    overlap = sum((a & b).values())
    if not a or not b:
        return float(not a and not b)
    return 2 * overlap / (sum(a.values()) + sum(b.values()))


def evaluate_generator(key: str, tokenizer, model, validation: pd.DataFrame = val_df):
    predictions = generate_answers(key, tokenizer, model, validation.Prompt.tolist())
    result = validation[["Master_Index", "Clinician"]].copy()
    result["prediction"] = predictions
    result["exact_match"] = [int(p.strip().lower() == r.strip().lower())
                             for p, r in zip(predictions, validation.Clinician.astype(str))]
    result["token_f1"] = [token_f1(p, r) for p, r in zip(predictions, validation.Clinician.astype(str))]
    path = OUTPUT_DIR / key / "validation_predictions.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(path, index=False)
    print(result[["exact_match", "token_f1"]].mean().to_dict())
    return result


def write_submission(key: str, tokenizer, model, test: pd.DataFrame = test_df):
    answers = generate_answers(key, tokenizer, model, test.Prompt.tolist())
    submission = pd.DataFrame({"Master_Index": test.Master_Index.tolist(), "Clinician": answers})
    path = OUTPUT_DIR / key / "submission.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, index=False)
    return path


# After training or loading a fine-tuned checkpoint:
# validation_results = evaluate_generator(MODEL_KEY, tokenizer, model)
# submission_path = write_submission(MODEL_KEY, tokenizer, model)

## Reopen a fine-tuned generator

The final directory contains a PEFT adapter or a full checkpoint (if LoRA was disabled). PEFT adapters identify their original base model in `adapter_config.json`. Use this helper for subsequent validation and test prediction runs.

In [ ]:
def load_finetuned_generator(key: str):
    checkpoint = OUTPUT_DIR / key / "final"
    if not checkpoint.is_dir():
        raise FileNotFoundError(f"No trained checkpoint: {checkpoint}")
    if (checkpoint / "adapter_config.json").exists():
        config = PeftConfig.from_pretrained(checkpoint)
        architecture = MODELS[key].architecture
        cls = AutoModelForSeq2SeqLM if architecture == "seq2seq" else AutoModelForCausalLM
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        base = cls.from_pretrained(config.base_model_name_or_path, token=HF_TOKEN, torch_dtype="auto")
        model = PeftModel.from_pretrained(base, checkpoint)
    else:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        cls = AutoModelForSeq2SeqLM if MODELS[key].architecture == "seq2seq" else AutoModelForCausalLM
        model = cls.from_pretrained(checkpoint, torch_dtype="auto")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.to(DEVICE)
    return tokenizer, model


# tokenizer, model = load_finetuned_generator(MODEL_KEY)
# evaluate_generator(MODEL_KEY, tokenizer, model)
# write_submission(MODEL_KEY, tokenizer, model)

## GatorTron: supervised panel features and answer retrieval

GatorTron is BERT-like; its classifier predicts the `Clinical Panel` category when available. Separately, mean-pooled embeddings retrieve a training-only `Clinician` answer for a new prompt. That response belongs to another case and may be clinically inappropriate. The baseline allows architectural comparison but requires human review; it does not synthesize a new answer. Both validation and test retrieval indices are built only from `train_df`.

In [ ]:
def train_gatortron(training: pd.DataFrame = train_df, validation: pd.DataFrame = val_df):
    if "Clinical Panel" not in training or "Clinical Panel" not in validation:
        raise ValueError("GatorTron classifier needs Clinical Panel labels")
    labels = sorted(training["Clinical Panel"].dropna().astype(str).unique())
    if not labels or not set(validation["Clinical Panel"].astype(str)).issubset(labels):
        raise ValueError("Validation has an unseen or missing Clinical Panel label")
    label_to_id = {name: idx for idx, name in enumerate(labels)}
    tokenizer = AutoTokenizer.from_pretrained(MODELS["gatortron"].model_id, token=HF_TOKEN)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODELS["gatortron"].model_id, num_labels=len(labels),
        id2label={i: label for i, label in enumerate(labels)}, label2id=label_to_id,
        token=HF_TOKEN,
    )
    def tokenize_frame(frame):
        ds = Dataset.from_pandas(pd.DataFrame({
            "text": frame.Prompt.astype(str).tolist(),
            "labels": frame["Clinical Panel"].astype(str).map(label_to_id).tolist(),
        }), preserve_index=False)
        return ds.map(lambda batch: tokenizer(batch["text"], truncation=True,
                                               max_length=min(MAX_SOURCE_TOKENS, 512)),
                      batched=True, remove_columns=["text"])
    from transformers import DataCollatorWithPadding
    run_dir = OUTPUT_DIR / "gatortron" / "classifier"
    trainer = Trainer(model=model, args=training_arguments(run_dir),
                      train_dataset=tokenize_frame(training), eval_dataset=tokenize_frame(validation),
                      data_collator=DataCollatorWithPadding(tokenizer), processing_class=tokenizer)
    trainer.train()
    metrics = trainer.evaluate()
    trainer.save_model(str(run_dir / "final"))
    tokenizer.save_pretrained(run_dir / "final")
    return tokenizer, model, metrics


def embed_prompts(tokenizer, encoder, prompts, batch_size=8):
    encoder.eval()
    device = next(encoder.parameters()).device
    vectors = []
    for start in range(0, len(prompts), batch_size):
        batch = tokenizer(list(prompts[start:start + batch_size]), return_tensors="pt",
                          padding=True, truncation=True, max_length=min(MAX_SOURCE_TOKENS, 512)).to(device)
        with torch.inference_mode():
            hidden = encoder(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            vectors.append(F.normalize(pooled.float(), dim=-1).cpu())
    return torch.cat(vectors, dim=0)


def predict_gatortron(tokenizer, encoder, reference: pd.DataFrame,
                      queries: pd.DataFrame) -> pd.DataFrame:
    if set(reference.Master_Index) & set(queries.Master_Index):
        raise ValueError("Reference and query IDs must not overlap")
    ref_vectors = embed_prompts(tokenizer, encoder, reference.Prompt.astype(str).tolist())
    query_vectors = embed_prompts(tokenizer, encoder, queries.Prompt.astype(str).tolist())
    nearest = (query_vectors @ ref_vectors.T).argmax(dim=1).tolist()
    return pd.DataFrame({"Master_Index": queries.Master_Index.tolist(),
                         "Clinician": reference.iloc[nearest].Clinician.tolist(),
                         "retrieved_from": reference.iloc[nearest].Master_Index.tolist()})


def evaluate_gatortron(tokenizer, encoder):
    result = predict_gatortron(tokenizer, encoder, train_df, val_df)
    result["token_f1"] = [token_f1(p, r) for p, r in
                          zip(result.Clinician, val_df.Clinician.astype(str))]
    print({"retrieval_token_f1": result.token_f1.mean()})
    result.to_csv(OUTPUT_DIR / "gatortron" / "validation_predictions.csv", index=False)
    return result


# tokenizer, classifier, metrics = train_gatortron()
# encoder = AutoModel.from_pretrained(MODELS["gatortron"].model_id, token=HF_TOKEN).to(DEVICE)
# retrieval_validation = evaluate_gatortron(tokenizer, encoder)
# retrieved_test = predict_gatortron(tokenizer, encoder, train_df, test_df)
# retrieved_test[["Master_Index", "Clinician"]].to_csv(OUTPUT_DIR / "gatortron" / "submission.csv", index=False)

## Optional teacher distillation and safe comparison

Med42 can generate candidate answers **only for the training split**. Keep the original clinician target by default; replacing it with pseudo-labels is an explicit experiment. Do not generate pseudo-labels for validation or competition test data. Model outputs may be inaccurate. Multi-head latent attention would require a separately implemented and evaluated architecture; it is not claimed here.

In [ ]:
def make_teacher_candidates(teacher_key="med42_v2", training: pd.DataFrame = train_df):
    tokenizer, model = load_model(teacher_key)
    model.to(DEVICE)
    answers = generate_answers(teacher_key, tokenizer, model, training.Prompt.tolist())
    candidates = training[["Master_Index", "Prompt", "Clinician"]].copy()
    candidates["teacher_candidate"] = answers
    path = OUTPUT_DIR / teacher_key / "teacher_candidates_train_only.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    candidates.to_csv(path, index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return candidates


# teacher_candidates = make_teacher_candidates()  # Requires access and enough memory.
# A reviewed subset can become an explicit training DataFrame; never substitute validation labels.

## Reproducibility and source links

- Record CSV provenance, consent/licensing and competition submission format before sharing or publishing data-derived outputs. CSV files are not included with this notebook.
- Reference model cards: [Med42](https://huggingface.co/m42-health/Llama3-Med42-8B), [BioMedLM](https://huggingface.co/stanford-crfm/BioMedLM), [ChatDoctor](https://huggingface.co/zl111/ChatDoctor), [BioGPT](https://huggingface.co/microsoft/biogpt), [Phi4-MedQA](https://huggingface.co/Vijayendra/Phi4-MedQA), [MediTron](https://huggingface.co/epfl-llm/meditron-7b), [Clinical T5](https://huggingface.co/hossboll/clinical-t5), [GatorTron](https://huggingface.co/UFNLP/gatortron-base), [Gemma 3](https://huggingface.co/google/gemma-3-1b-it).
- [Google Med-PaLM research page](https://sites.research.google/med-palm/) describes its access context. The unrelated `medpalm` PyPI package is not the published Google model weights.
- Practical caveat: the Phi4-MedQA adapter currently declares a quantized `unsloth/phi-4-unsloth-bnb-4bit` base. Check compatible CUDA/bitsandbytes support before this run. Nothing here asserts that checkpoints were successfully downloaded or trained.
- Clear memory between runs: `del model; gc.collect(); torch.cuda.empty_cache()` when CUDA is available.